# Mini Project 1 -- How Film Dialogue Reflects the Human-Technology Relationship

**Name:** Riti Upadhyay  
**Dataset:** Cornell Movie Dialogs Corpus  
**Date:** May 2026

---

## Section 1 -- Overview

**Dataset:** Cornell Movie Dialogs Corpus. It contains 220,579 conversational exchanges from 617 movie scripts. Each line of dialogue includes the movie title, release year, and genre. The data covers films from the 1930s through the 2010s. Source: https://convokit.cornell.edu/documentation/movie.html

**Why this dataset:** Film dialogue is one of the few places where you can track how ordinary people talked about technology over eight decades without needing surveys or interviews.

**Three analytical questions:**

1. Which technology terms appear first in film dialogue and how does their frequency change across decades? Does communication technology follow a different growth curve than computational technology?
2. Which emotion-related words appear near technology terms in dialogue, and how does that vocabulary shift across eras? Ordinary high-frequency words (know, use, time, call) are excluded so the analysis tracks affect and attitudes toward technology, not random co-occurrence noise.
3. Do communication technology terms and computational technology terms shift toward sentence-initial position at different rates across decades? Sentence-initial position is used as a proxy for grammatical agency.

**Note on how the questions changed:** The original Q2 proposed full sentiment scoring, which requires tools beyond pandas. It was reframed as **emotion-keyword neighbor** analysis: only words from a fixed affect/attitude lexicon count near tech terms, so we study how the emotional register around technology in dialogue evolves. The original Q3 asked about grammatical subject vs object, which requires dependency parsing. It was reframed as a sentence position proxy, and expanded to compare two categories of technology terms rather than treating all tech terms as one group. This makes it possible to answer without NLP tools beyond what was covered in class.

**What a practitioner would do with these findings:** A UX researcher or interaction designer could use Q1’s timelines and Q3’s agency proxy to see when different kinds of technology entered everyday film dialogue. Q2’s **emotion-keyword** lens helps explain how characters (and audiences) *talked about* tech in affective terms—worry, trust, anger, hope—rather than being misled by high-frequency but emotionally empty neighbors like *know* or *use*. Together, that clarifies cultural assumptions people already carried before HCI began designing for them.

In [1]:
# Setup -- run this cell first
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px
import re
from collections import Counter
from pathlib import Path

print("Setup complete.")

Setup complete.


---

## Section 2 -- Data Profile

We load three CSV files generated by `fetch_cornell.py`. The fetch script downloads the Cornell corpus zip, parses the custom separator format, joins dialogue lines with movie metadata, and saves the results. No manual file handling is needed. **Question 2** in this notebook then reads the `text` column from `cornell_clean.csv` (lines that contain technology terms) and applies a **curated emotion lexicon** defined in the **Q2 code cell**; emotion neighbor analysis is not part of `fetch_cornell.py`; the script only tags tech terms and writes the CSVs.

- `cornell_clean.csv` -- one row per dialogue line, the full cleaned dataset
- `cornell_tidy.csv` -- tidy format, one row per technology term mention per line
- `cornell_term_by_decade.csv` -- summary, term frequency normalized per 1000 lines per decade

In [2]:
# Load all three datasets.
df = pd.read_csv("cornell_clean.csv")
tidy_df = pd.read_csv("cornell_tidy.csv")
term_df = pd.read_csv("cornell_term_by_decade.csv")

print("cornell_clean.csv shape:", df.shape)
print("cornell_tidy.csv shape:", tidy_df.shape)
print("cornell_term_by_decade.csv shape:", term_df.shape)
df.head()

cornell_clean.csv shape: (304160, 11)
cornell_tidy.csv shape: (5535, 8)
cornell_term_by_decade.csv shape: (112, 6)


,movie_id,title,year,decade,genres,text,has_tech,tech_terms_found,first_tech_position,first_tech_category,line_length
0,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",They do not!,False,NaN,NaN,NaN,3
1,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",They do to!,False,NaN,NaN,NaN,3
2,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",I hope so.,False,NaN,NaN,NaN,3
3,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",She okay?,False,NaN,NaN,NaN,2
4,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",Let's go.,False,NaN,NaN,NaN,3


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 304160 entries, 0 to 304159
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   movie_id             304160 non-null  str    
 1   title                304160 non-null  str    
 2   year                 304160 non-null  int64  
 3   decade               304160 non-null  int64  
 4   genres               304160 non-null  str    
 5   text                 304160 non-null  str    
 6   has_tech             304160 non-null  bool   
 7   tech_terms_found     5269 non-null    str    
 8   first_tech_position  5269 non-null    float64
 9   first_tech_category  5269 non-null    str    
 10  line_length          304160 non-null  int64  
dtypes: bool(1), float64(1), int64(3), str(6)
memory usage: 23.5 MB


In [4]:
df.describe()

,year,decade,first_tech_position,line_length
count,304160.000000,304160.000000,5269.000000,304160.000000
mean,1988.164831,1983.495759,0.494345,11.210060
std,17.046154,17.101330,0.275796,12.632913
min,1931.000000,1930.000000,0.000000,0.000000
25%,1984.000000,1980.000000,0.264368,4.000000
50%,1995.000000,1990.000000,0.505102,7.000000
75%,1999.000000,1990.000000,0.729730,14.000000
max,2010.000000,2010.000000,0.989655,582.000000


In [5]:
# Check for missing values.
# first_tech_position and first_tech_category will be null
# for lines that contain no technology terms. That is expected.
df.isnull().sum()

movie_id                    0
title                       0
year                        0
decade                      0
genres                      0
text                        0
has_tech                    0
tech_terms_found       298891
first_tech_position    298891
first_tech_category    298891
line_length                 0
dtype: int64

**Data profile notes:**

The full dataset has one row per dialogue line. The key columns for analysis are `decade`, `has_tech`, `tech_terms_found`, `first_tech_position`, and `first_tech_category`. The `first_tech_position` and `first_tech_category` columns are null for lines with no technology terms. That is expected and those rows are excluded from the Q3 analysis. The tidy dataset restructures the `tech_terms_found` column so each term gets its own row with its own `category` column. This is the tidy data structure from class: one observation per row, one variable per column. The summary dataset normalizes term counts by total lines per decade so decades with more films do not dominate the results.

---

## Section 3 -- Analysis

Question 2’s **emotion lexicon** and `get_emotion_neighbors()` are defined at the **top of the Question 2 Python cell** (under the Q2 heading), so that cell runs on its own after you load the CSVs. Q1 uses only the loaded CSVs. As noted by the professor, you do not need to paste the same helper into every question block—here the Q2 cell holds the full definition once.

**Reminder:** Scroll to **Question 2** and run its Python cell from the top—the first block in that cell is the lexicon and `get_emotion_neighbors()`; the rest builds the charts.


**Question 1:** Which technology terms appear first in film dialogue and how does their frequency change across decades? Does communication technology follow a different growth curve than computational technology?

In [9]:
# Filter to the six most meaningful terms, three per category.
# These were chosen because they represent distinct technologies
# and have enough mentions across decades to show a real trend.
terms_to_plot = ["phone", "radio", "television", "computer", "internet", "ai"]

q1_df = term_df[term_df["term"].isin(terms_to_plot)].copy()
q1_df = q1_df[q1_df["decade"] >= 1930]

# Line chart: each line is one term, x axis is decade, y axis is frequency.
# Line chart is the right choice here because the question is about
# change over time across many periods.
fig1 = px.line(
    q1_df,
    x="decade",
    y="mentions_per_1000",
    color="term",
    title="Communication Tech Entered Film Dialogue Decades Before Computational Tech",
    labels={
        "decade": "Decade",
        "mentions_per_1000": "Mentions per 1,000 Lines",
        "term": "Technology Term"
    },
    markers=True
)
fig1.update_layout(xaxis=dict(dtick=10))
fig1.write_image("chart1_tech_frequency_by_decade.png")
fig1.show()

**Interpretation:** The chart shows when each technology term entered film dialogue and how its frequency grew. Radio and phone appear earliest, likely from the 1930s onward. Computer and internet appear later, growing sharply in the 1980s and 1990s. The two categories do follow different curves. Communication technology entered the cultural vocabulary first and stayed consistently present. Computational technology entered later but its growth rate accelerated faster once it appeared. This matters for HCI because it means film characters were already navigating a technology-saturated world through communication devices long before computers became part of the conversation.

**Question 2:** Which **emotion-related** words appear near technology terms in film dialogue, and how does that vocabulary shift between the pre-1990 and post-1990 corpora? Analysis uses a **curated affect/attitude lexicon** in the Q2 code cell below (not all neighbors), and reports rates **per 1,000 dialogue lines that contain a technology term** so the two eras are comparable.

In [11]:
# Curated affect / attitude vocabulary for Q2. Neighbors of tech terms are counted
# only if they appear in this set—so we study emotion around technology, not
# generic glue words (know, use, time, said, call, make, right, help, etc.).
EMOTION_LEXICON = frozenset(
    """
    love loved loves loving hate hated hates dislike disliked
    like liked likes unlike
    miss missed missing
    scared scary scares fear feared fears afraid frightening frightened
    worried worry worries worrying
    hope hoped hopes hoping hopeless hopeful
    happy happily happier happiness unhappy sadness sad sadly sadder angrier angry madder mad upset upsets
    trust trusted trusting distrust distrusted
    lie lied lies lying honest honesty truthful truth
    terrible terribly awful horribly horrible bad worse worst
    good better best great greater greatly amazing amazed wonder wonderful
    nicely nice sweeter sweet dearest dear darling precious
    evil wicked creep creeps creepy weird weirder strangest strange oddest odd
    funny funnier funniest fun
    dumb stupid stupidity idiot idiots insane insanity crazy craziest
    sick sicker safest safe safely danger dangerous endangered
    hurt hurts hurting painfully painful pain
    sorry shame ashamed shameful proud proudly
    disgusting disgusted gross
    beautiful beautifully ugliness ugly uglier
    lonely lonelier alone helpless
    depressing depressed depression
    anxious anxiety anxiously nervous nervously calm calmer relaxed relaxing
    stress stressful stressed
    surprised surprising shock shocked shocking
    exciting excited excitement boring bored
    disappointed disappointing disappointment
    glad gladly gratitude grateful appreciate appreciated
    cruel cruelty kind kindly kindness gentle gentler meanness mean
    jealous jealousy envy envious guilty guilt innocence innocent blame blamed forgive forgiven
    panic panicked terrifying terror horrified horrifying nightmare dream dreams dreamed
    pathetic pity disaster disastrous hopeless hopelessness stunning stunningly brilliant
    fool foolish genius disgust joy joyous miserable misery horror horrors frighten
    appalling appalled outrage outraged furious fury rage
    feel felt feeling feelings
    betrayed betrayal betray adore adored adores
    """.split()
)


def get_emotion_neighbors(text, term, window=5):
    """Words within a window of a tech term that appear in EMOTION_LEXICON only."""
    words = re.findall(r"\b\w+\b", str(text).lower())
    found = []
    for i, word in enumerate(words):
        if word != term:
            continue
        start = max(0, i - window)
        end = min(len(words), i + window + 1)
        neighbors = words[start:i] + words[i + 1 : end]
        for w in neighbors:
            if w in EMOTION_LEXICON:
                found.append(w)
    return found


print("Emotion-neighbor helper defined.")

# Split into pre-1990 and post-1990 periods.
# Top emotion lexicon hits near tech terms, normalized per 1,000 tech-containing lines.
tech_lines = df[df["has_tech"]].copy()
early = tech_lines[tech_lines["decade"] < 1990]
late = tech_lines[tech_lines["decade"] >= 1990]

# Same technology tokens as fetch_cornell.py (word-boundary match, not substring).
FOCUS_TERMS = [
    "phone", "telephone", "radio", "television", "tv", "call", "signal",
    "computer", "internet", "online", "email", "app", "algorithm",
    "ai", "robot", "digital", "machine", "screen", "device", "network", "data",
]


def top_emotion_words(subset, n=15):
    all_words = []
    for _, row in subset.iterrows():
        text = str(row["text"])
        text_lower = text.lower()
        for term in FOCUS_TERMS:
            if re.search(rf"\b{re.escape(term)}\b", text_lower):
                all_words.extend(get_emotion_neighbors(text, term))
    n_lines = max(len(subset), 1)
    counts = Counter(all_words)
    ranked = counts.most_common(n)
    return [(w, c / n_lines * 1000.0) for w, c in ranked]


print("Computing emotion-keyword rates for pre-1990 lines...")
early_words = top_emotion_words(early)
print("Computing emotion-keyword rates for post-1990 lines...")
late_words = top_emotion_words(late)

early_df = pd.DataFrame(early_words, columns=["word", "per_1000_tech_lines"])
late_df = pd.DataFrame(late_words, columns=["word", "per_1000_tech_lines"])

fig2a = px.bar(
    early_df.sort_values("per_1000_tech_lines"),
    x="per_1000_tech_lines",
    y="word",
    orientation="h",
    title="Emotion-Related Words Near Tech Terms -- Pre-1990 (per 1,000 tech lines)",
    labels={
        "per_1000_tech_lines": "Emotion keyword mentions / 1,000 tech lines",
        "word": "Emotion keyword",
    },
)
fig2a.write_image("chart2a_emotion_near_tech_pre1990.png")
fig2a.show()

fig2b = px.bar(
    late_df.sort_values("per_1000_tech_lines"),
    x="per_1000_tech_lines",
    y="word",
    orientation="h",
    title="Emotion-Related Words Near Tech Terms -- Post-1990 (per 1,000 tech lines)",
    labels={
        "per_1000_tech_lines": "Emotion keyword mentions / 1,000 tech lines",
        "word": "Emotion keyword",
    },
)
fig2b.write_image("chart2b_emotion_near_tech_post1990.png")
fig2b.show()

Computing emotion-keyword rates for pre-1990 lines...


NameError: name 'get_emotion_neighbors' is not defined

**Interpretation:** Each chart lists the most frequent **emotion or attitude** words that fall within a short window of any technology term, after restricting counts to a **fixed lexicon** in the notebook. That deliberately drops generic dialogue words (know, use, call, time, help) that dominated earlier “any neighbor word” tallies but do not indicate how characters *feel* about technology. The horizontal axis is **mentions per 1,000 lines that contain a tech term**, so pre-1990 and post-1990 are comparable even though the corpus has more dialogue in later decades. Compare the two charts: a relative rise in fear-, trust-, love-, or anxiety-type words in the later period supports a story about **shifting emotional framing** around tech—not merely more films. The limitation is that a keyword list cannot capture irony or negation (“not afraid” still registers *afraid* as a token), so these charts show **lexical co-presence**, not full sentiment.

**Question 3:** Do communication technology terms and computational technology terms shift toward sentence-initial position at different rates across decades?

In [ ]:
# Use first_tech_position (0 to 1 scale) and first_tech_category.
# Position below 0.33 means the term appears in the first third
# of the line, which we treat as a proxy for subject position.
# We compute the percentage of lines where this is true,
# grouped by decade and category.

tech_pos = df[df["first_tech_position"].notna()].copy()
tech_pos["is_early"] = tech_pos["first_tech_position"] < 0.33

# Group by decade and category.
total_by_group = tech_pos.groupby(["decade", "first_tech_category"]).size().reset_index(name="total")
early_by_group = tech_pos[tech_pos["is_early"]].groupby(["decade", "first_tech_category"]).size().reset_index(name="early_count")

q3_df = total_by_group.merge(early_by_group, on=["decade", "first_tech_category"], how="left")
q3_df["early_count"] = q3_df["early_count"].fillna(0)
q3_df["pct_early"] = (q3_df["early_count"] / q3_df["total"]) * 100
q3_df = q3_df[q3_df["decade"] >= 1930]

# Faceted line chart: one line per category, showing the percentage
# of lines where the term appears in the first third of the sentence.
# Faceting by category (communication vs computational) shows
# whether the two types of technology follow different agency trajectories
# without any manual classification of individual words.
fig3 = px.line(
    q3_df,
    x="decade",
    y="pct_early",
    color="first_tech_category",
    title="Computational Tech Shifts to Subject Position Later Than Communication Tech",
    labels={
        "decade": "Decade",
        "pct_early": "% of Lines Where Term Appears Early (Subject Proxy)",
        "first_tech_category": "Technology Category"
    },
    markers=True
)
fig3.update_layout(xaxis=dict(dtick=10))
fig3.write_image("chart3_agency_by_category.png")
fig3.show()

**Interpretation:** The chart shows the percentage of tech-containing lines where the technology term appears in the first third of the sentence, split by communication technology versus computational technology. If the computational category line rises in recent decades and pulls away from the communication category line, it means computational tech is increasingly framed as the subject of sentences rather than the object. That is the agency shift the question is looking for. Communication technology like radio and phone may have been in subject position earlier simply because it was older and more embedded in everyday speech. Computational technology becoming agentive later maps onto real cultural moments like the rise of the internet and algorithmic systems.

---

## Section 4 -- Conclusions

**Summary of findings:**

Across three lenses, the analysis tracks how film dialogue has encoded the human-technology relationship over eight decades. The first question established the timeline: communication technology entered film vocabulary first and stayed consistently present, while computational technology appeared later but grew faster. The second question asked which **emotion-related** words cluster near technology terms in each era, using a curated lexicon and normalized rates so we interpret change in affective language—not accidental high-frequency neighbors. The third question asked whether technology terms are increasingly treated as grammatical subjects rather than objects, and whether that shift happens at different rates for communication and computational categories. Together the three questions make a connected argument about how technology is culturally framed in dialogue over time. The main limitation is that sentence position is an imperfect proxy for grammatical subject. The corpus also skews toward English-language Hollywood films, so the findings reflect one cultural context. The most interesting next step would be to split the analysis by genre to see whether science fiction films encode computational technology as agentive earlier than drama does.